In [33]:
import torch
torch.cuda.device_count()

1

In [34]:
import pandas as pd
df = pd.read_excel("/scratch/prj0000000262/Sanwa_forecast_new/100T-8 Parameter Data Y2025 to Y2026 Feb for AI Forecast.xlsx")

In [35]:
# 1) ensure it's a real datetime (important if it's currently a string)
df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])

# 2) sort oldest -> latest
df = df.sort_values(by=df.columns[0], ascending=True).reset_index(drop=True)

In [36]:
cols = [
    "injection_time",
    "switchover_position",
    "max_injection_pressure",
    "switchover_pressure",
    "end_of_packing_stroke",
    "plastification_time",
    "nozzle_temperature",
    "barrel_front_temperature",
    "barrel_center_temperature",
    "barrel_rear_temperature",
    "feeder_temperature",
]

df = df[cols]
df

,injection_time,switchover_position,max_injection_pressure,switchover_pressure,end_of_packing_stroke,plastification_time,nozzle_temperature,barrel_front_temperature,barrel_center_temperature,barrel_rear_temperature,feeder_temperature
0,1.69,10.00,136.1,991,1.52,7.37,254.7,255.3,255.3,244.3,59.1
1,1.68,10.00,135.3,1035,2.17,7.36,255.4,255.1,255.2,243.7,59.7
2,1.69,10.00,139.3,1055,2.27,7.45,254.8,255.1,255.1,243.7,59.9
3,1.68,10.00,142.1,1067,1.96,7.52,255.1,255.1,254.9,243.8,59.1
4,1.69,10.00,144.3,1072,2.41,7.74,255.1,255.0,254.8,243.9,59.1
...,...,...,...,...,...,...,...,...,...,...,...
617425,1.83,8.00,160.4,1243,4.11,8.52,255.0,254.9,254.8,244.9,59.8
617426,1.83,7.99,159.8,1245,4.06,8.53,255.0,254.9,254.9,245.0,60.2
617427,1.83,7.99,160.1,1243,4.09,8.59,254.9,255.0,254.8,245.0,59.5
617428,1.83,8.00,160.0,1244,4.15,8.59,255.1,255.1,254.9,245.0,59.0


In [37]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Your 12 channels in order
cols = [
    "injection_time",
    "switchover_position",
    "max_injection_pressure",
    "switchover_pressure",
    "end_of_packing_stroke",
    "plastification_time",
    "nozzle_temperature",
    "barrel_front_temperature",
    "barrel_center_temperature",
    "barrel_rear_temperature",
    "feeder_temperature",
]

def write_one_step_windows_to_parquet(
    df: pd.DataFrame,
    feature_cols,
    out_path="one_step_ws1024.parquet",
    window_size=1024,
    batch_size=4096,
    compression="zstd",
    sort_by_time_col=None,  # e.g. "immac.create_time" if you still have it
):
    # 1) optional sort
    if sort_by_time_col is not None and sort_by_time_col in df.columns:
        df = df.sort_values(sort_by_time_col)

    # 2) keep only needed columns + float32
    df = df.loc[:, feature_cols].dropna()
    arr = df.to_numpy(dtype=np.float32, copy=False)  # shape: (N, 12)

    N, C = arr.shape
    if C != 11:
        raise ValueError(f"Expected 11 channels, got {C}")
    if N <= window_size:
        raise ValueError(f"Need N > window_size. Got N={N}, window_size={window_size}")

    num_samples = N - window_size  # because y is at t+window_size
    writer = None

    for start in range(0, num_samples, batch_size):
        end = min(start + batch_size, num_samples)
        count = end - start

        # chunk includes enough future to build windows
        chunk = arr[start : end + window_size]  # shape: (count + window_size, 12)

        # windows over time axis -> (count+1, window_size, 12), drop last (no y for it)
        Xw = np.lib.stride_tricks.sliding_window_view(
            chunk, window_shape=window_size, axis=0
        )[:count]

        y = arr[start + window_size : end + window_size]  # shape: (count, 12)

        # store X as fixed-size list of length (1024*12) per row
        x_flat = Xw.reshape(count, window_size * C)
        x_arrow = pa.FixedSizeListArray.from_arrays(
            pa.array(x_flat.reshape(-1), type=pa.float32()),
            list_size=window_size * C,
        )

        # store y as fixed-size list of length 12 per row
        y_arrow = pa.FixedSizeListArray.from_arrays(
            pa.array(y.reshape(-1), type=pa.float32()),
            list_size=C,
        )

        table = pa.Table.from_arrays([x_arrow, y_arrow], names=["samples", "labels"])

        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression=compression)

        writer.write_table(table)

    if writer is not None:
        writer.close()

    return out_path

out_file = write_one_step_windows_to_parquet(
    df=df,
    feature_cols=cols,
    out_path="one_step_ws1024.parquet",
    window_size=1024,
    batch_size=4096,
    compression="zstd",
    sort_by_time_col=None,  # or "immac.create_time"
)
print("Saved:", out_file)

Saved: one_step_ws1024.parquet


In [38]:
import pandas as pd
df = pd.read_parquet("one_step_ws1024.parquet")
df

,samples,labels
0,"[1.69, 1.68, 1.69, 1.68, 1.69, 1.69, 1.69, 1.6...","[1.61, 10.0, 154.4, 1141.0, 3.03, 7.78, 255.8,..."
1,"[1.68, 1.69, 1.68, 1.69, 1.69, 1.69, 1.68, 1.6...","[1.61, 10.0, 151.8, 1132.0, 3.04, 7.42, 254.8,..."
2,"[1.69, 1.68, 1.69, 1.69, 1.69, 1.68, 1.69, 1.6...","[1.61, 9.99, 151.4, 1129.0, 3.13, 7.46, 255.1,..."
3,"[1.68, 1.69, 1.69, 1.69, 1.68, 1.69, 1.69, 1.6...","[1.6, 9.99, 151.4, 1121.0, 3.29, 7.5, 255.0, 2..."
4,"[1.69, 1.69, 1.69, 1.68, 1.69, 1.69, 1.69, 1.6...","[1.61, 10.0, 153.2, 1132.0, 3.22, 7.59, 255.2,..."
...,...,...
616401,"[1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.84, 1.8...","[1.83, 8.0, 160.4, 1243.0, 4.11, 8.52, 255.0, ..."
616402,"[1.83, 1.83, 1.83, 1.83, 1.83, 1.84, 1.83, 1.8...","[1.83, 7.99, 159.8, 1245.0, 4.06, 8.53, 255.0,..."
616403,"[1.83, 1.83, 1.83, 1.83, 1.84, 1.83, 1.83, 1.8...","[1.83, 7.99, 160.1, 1243.0, 4.09, 8.59, 254.9,..."
616404,"[1.83, 1.83, 1.83, 1.84, 1.83, 1.83, 1.83, 1.8...","[1.83, 8.0, 160.0, 1244.0, 4.15, 8.59, 255.1, ..."


In [39]:
df['labels'].iloc[0].shape

(11,)

In [40]:
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from pathlib import Path

def split_parquet_random_streaming_notebook(
    input_path: str,
    out_dir: str,
    train_ratio: float = 0.6,
    val_ratio: float = 0.2,
    test_ratio: float = 0.2,
    seed: int = 42,
    batch_size: int = 8192,
    compression: str = "zstd",
):
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9, "Ratios must sum to 1."

    input_path = Path(input_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_path = out_dir / "train_1p.parquet"
    val_path   = out_dir / "val.parquet"
    test_path  = out_dir / "test.parquet"

    pf = pq.ParquetFile(str(input_path))
    schema = pf.schema_arrow

    train_writer = pq.ParquetWriter(str(train_path), schema=schema, compression=compression)
    val_writer   = pq.ParquetWriter(str(val_path),   schema=schema, compression=compression)
    test_writer  = pq.ParquetWriter(str(test_path),  schema=schema, compression=compression)

    rng = np.random.default_rng(seed)
    t1 = train_ratio
    t2 = train_ratio + val_ratio

    n_train = n_val = n_test = 0

    try:
        for rb in pf.iter_batches(batch_size=batch_size):
            n = rb.num_rows
            r = rng.random(n)

            m_train = pa.array(r < t1)
            m_val   = pa.array((r >= t1) & (r < t2))
            m_test  = pa.array(r >= t2)

            tbl = pa.Table.from_batches([rb])

            if pc.any(m_train).as_py():
                train_tbl = pc.filter(tbl, m_train)
                train_writer.write_table(train_tbl)
                n_train += train_tbl.num_rows

            if pc.any(m_val).as_py():
                val_tbl = pc.filter(tbl, m_val)
                val_writer.write_table(val_tbl)
                n_val += val_tbl.num_rows

            if pc.any(m_test).as_py():
                test_tbl = pc.filter(tbl, m_test)
                test_writer.write_table(test_tbl)
                n_test += test_tbl.num_rows

    finally:
        train_writer.close()
        val_writer.close()
        test_writer.close()

    total = n_train + n_val + n_test
    print("Done splitting.")
    print(f"train: {n_train} ({n_train/total:.2%})")
    print(f"val:   {n_val} ({n_val/total:.2%})")
    print(f"test:  {n_test} ({n_test/total:.2%})")
    print("Saved to:", out_dir)
    return str(train_path), str(val_path), str(test_path)

# Example:
train_p, val_p, test_p = split_parquet_random_streaming_notebook(
    input_path="one_step_ws1024.parquet",
    out_dir="splits",
    seed=123,
    batch_size=8192
)

Done splitting.
train: 370118 (60.04%)
val:   123280 (20.00%)
test:  123008 (19.96%)
Saved to: splits


In [41]:
import pandas as pd
df_test = pd.read_parquet("/scratch/prj0000000262/Sanwa_forecast_new/test.parquet")
df_test

,samples,labels
0,"[1.69, 1.69, 1.68, 1.69, 1.69, 1.69, 1.69, 1.6...","[1.61, 10.0, 153.0, 1120.0, 3.01, 7.62, 254.8,..."
1,"[1.69, 1.68, 1.69, 1.69, 1.69, 1.69, 1.68, 1.6...","[1.61, 9.99, 153.7, 1125.0, 3.14, 7.66, 255.0,..."
2,"[1.69, 1.69, 1.69, 1.69, 1.68, 1.69, 1.69, 1.6...","[1.61, 10.0, 153.1, 1134.0, 3.12, 7.73, 255.2,..."
3,"[1.69, 1.69, 1.69, 1.68, 1.69, 1.69, 1.68, 1.6...","[1.61, 9.99, 152.7, 1125.0, 3.08, 7.78, 255.0,..."
4,"[1.68, 1.69, 1.69, 1.68, 1.69, 1.68, 1.69, 1.6...","[1.61, 10.0, 153.8, 1132.0, 3.08, 7.75, 255.1,..."
...,...,...
123003,"[1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.8...","[1.83, 7.99, 160.6, 1247.0, 4.28, 8.39, 254.8,..."
123004,"[1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.8...","[1.83, 8.0, 160.7, 1247.0, 4.36, 8.4, 254.8, 2..."
123005,"[1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.83, 1.8...","[1.85, 7.99, 158.8, 1231.0, 4.15, 8.5, 255.0, ..."
123006,"[1.83, 1.83, 1.84, 1.83, 1.83, 1.83, 1.84, 1.8...","[1.83, 8.0, 161.5, 1249.0, 4.39, 8.41, 254.9, ..."


In [42]:
df_test['samples'].iloc[0].shape

(11264,)

In [43]:
df_test['samples'].iloc[0].reshape(11,-1)

array([[  1.69,   1.69,   1.68, ...,   1.61,   1.6 ,   1.61],
       [ 10.  ,  10.  ,   9.99, ...,   9.99,   9.99,  10.  ],
       [145.2 , 146.  , 145.8 , ..., 151.4 , 151.4 , 153.2 ],
       ...,
       [254.8 , 254.7 , 254.7 , ..., 255.2 , 255.1 , 255.  ],
       [243.9 , 244.1 , 244.4 , ..., 245.1 , 244.8 , 244.7 ],
       [ 59.5 ,  59.9 ,  59.9 , ...,  59.9 ,  59.9 ,  59.1 ]],
      dtype=float32)

In [44]:
df_test['labels'].iloc[0].shape

(11,)

# Prediction and Plot

In [45]:
## recreating the train, val, test membership
import numpy as np
import pyarrow.parquet as pq

# 0=train, 1=val, 2=test
def reconstruct_split_mask(
    input_path,
    train_ratio=0.6,
    val_ratio=0.2,
    test_ratio=0.2,
    seed=123,
    batch_size=8192,   # can be different; RNG stream still consistent as long as row order is same
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9
    pf = pq.ParquetFile(str(input_path))
    N = pf.metadata.num_rows

    rng = np.random.default_rng(seed)
    t1 = train_ratio
    t2 = train_ratio + val_ratio

    split = np.empty(N, dtype=np.uint8)

    idx = 0
    # read minimal columns just to iterate row counts reliably
    for rb in pf.iter_batches(batch_size=batch_size, columns=["labels"]):
        n = rb.num_rows
        r = rng.random(n)
        split[idx:idx+n] = np.where(r < t1, 0, np.where(r < t2, 1, 2))
        idx += n

    assert idx == N
    return split

split_mask = reconstruct_split_mask("one_step_ws1024.parquet", seed=123)
print("counts:", np.bincount(split_mask))

counts: [370118 123280 123008]


In [46]:
split_mask.shape

(616406,)

In [47]:
import matplotlib.pyplot as plt
import numpy as np

def plot_chrono_true_pred(
    y_true, y_pred, split_mask,
    channel=0,
    max_points=50000,
    title=None,
):
    N = y_true.shape[0]
    stride = max(1, N // max_points)
    idx = np.arange(0, N, stride)

    yt = y_true[idx, channel]
    yp = y_pred[idx, channel]
    sm = split_mask[idx]

    plt.figure(figsize=(16, 5))
    plt.plot(idx, yt, label="True", linewidth=1)
    plt.plot(idx, yp, label="Pred", linewidth=1)

    # mark split membership as a thin scatter at bottom
    y0 = np.nanmin(yt) - 0.02*(np.nanmax(yt)-np.nanmin(yt) + 1e-9)
    for lab, name in [(0, "train"), (1, "val"), (2, "test")]:
        m = (sm == lab)
        plt.scatter(idx[m], np.full(m.sum(), y0), s=3, alpha=0.25, label=f"{name} (marker)")

    plt.legend()
    plt.xlabel("Chronological row index")
    plt.ylabel(f"Value (channel {channel})")
    plt.title(title or f"Chronological True vs Pred (ch={channel})")
    plt.tight_layout()
    plt.show()

In [58]:
pred_test = np.load("/scratch/prj0000000262/Sanwa_forecast_new/Sanwa_forecast_ft/checkpoints/FT_tiny_cavity_1_from1_Sw_fc_bs8_lr0.0001_seed42_xnormzscore_ynormzscore_20260313_162140/test_preds.npy")     # shape [N_test, 12]
true_test = np.load("/scratch/prj0000000262/Sanwa_forecast_new/Sanwa_forecast_ft/checkpoints/FT_tiny_cavity_1_from1_Sw_fc_bs8_lr0.0001_seed42_xnormzscore_ynormzscore_20260313_162140/test_targets.npy")


In [59]:
pred_test.shape

(15376, 11)

In [55]:
import numpy as np

test_idx = np.where(split_mask == 2)[0]       # chronological positions in original file

pred_test = np.load("/scratch/prj0000000262/Sanwa_forecast_new/Sanwa_forecast_ft/checkpoints/FT_tiny_Sanwa_forecast_new_from1_Sw_fc_bs8_lr0.0001_seed42_xnormzscore_ynormzscore_20260313_153523/test_preds.npy")     # shape [N_test, 12]
true_test = np.load("/scratch/prj0000000262/Sanwa_forecast_new/Sanwa_forecast_ft/checkpoints/FT_tiny_Sanwa_forecast_new_from1_Sw_fc_bs8_lr0.0001_seed42_xnormzscore_ynormzscore_20260313_153523/test_targets.npy")

# Create full-length arrays with NaN except test points
N = split_mask.shape[0]
C = pred_test.shape[1]
pred_full = np.full((N, C), np.nan, dtype=np.float32)
true_full = np.full((N, C), np.nan, dtype=np.float32)

pred_full[test_idx] = pred_test
true_full[test_idx] = true_test

# Plot test points in chronological context (train/val are NaN in line)
plot_chrono_true_pred(true_full, pred_full, split_mask, channel=1, max_points=80000,
                      title="TEST predictions placed back into chronological timeline")

ValueError: shape mismatch: value array of shape (15376,11) could not be broadcast to indexing result of shape (123008,11)

In [52]:
import pyarrow.parquet as pq
import numpy as np

orig_parquet = "one_step_ws1024.parquet"   # <-- the parquet you used in your split script
test_parquet = "/scratch/prj0000000262/Sanwa_forecast_new/train_1p.parquet"  # <-- the one used in training

N_orig = pq.ParquetFile(orig_parquet).metadata.num_rows
N_test_file = pq.ParquetFile(test_parquet).metadata.num_rows

print("N_orig:", N_orig)
print("N_test_file:", N_test_file)
print("pred_test rows:", pred_test.shape[0])
print("len(test_idx):", len(test_idx))

N_orig: 616406
N_test_file: 370118
pred_test rows: 15376
len(test_idx): 123008


In [53]:
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, HTML

def plot_chrono_true_pred_interactive(true_full, pred_full, split_mask, channel=0, max_points=200_000, title=""):
    """
    true_full/pred_full: (N, C) with NaNs except at test indices (or wherever you filled)
    split_mask: (N,) with values {0=train,1=val,2=test}
    channel: which output channel to plot
    max_points: downsample cap for responsiveness
    """
    N = true_full.shape[0]
    idx_all = np.arange(N)

    y_true = true_full[:, channel]
    y_pred = pred_full[:, channel]

    # Downsample uniformly for speed
    stride = max(1, N // max_points)
    idx = idx_all[::stride]
    yt = y_true[::stride]
    yp = y_pred[::stride]
    sm = split_mask[::stride]

    # Keep only points where at least one is not NaN (so lines don't have huge gaps cost)
    keep = ~(np.isnan(yt) & np.isnan(yp))
    idx = idx[keep]
    yt = yt[keep]
    yp = yp[keep]
    sm = sm[keep]

    # Marker baseline (place split markers slightly below the visible min)
    finite_vals = yt[np.isfinite(yt)]
    if finite_vals.size == 0:
        raise ValueError("No finite true values found in this channel after filtering.")
    y_min = float(np.nanmin(finite_vals))
    y_max = float(np.nanmax(finite_vals))
    y0 = y_min - 0.03 * (y_max - y_min + 1e-9)

    fig = go.Figure()

    # True line
    fig.add_trace(go.Scattergl(
        x=idx, y=yt,
        mode="lines",
        name="True",
        connectgaps=False,
    ))

    # Pred line
    fig.add_trace(go.Scattergl(
        x=idx, y=yp,
        mode="lines",
        name="Pred",
        connectgaps=False,
    ))

    # Split markers (train/val/test)
    for lab, name in [(0, "train"), (1, "val"), (2, "test")]:
        m = (sm == lab)
        fig.add_trace(go.Scattergl(
            x=idx[m],
            y=np.full(m.sum(), y0),
            mode="markers",
            name=f"{name} (marker)",
            marker=dict(size=4),
            opacity=0.35,
        ))

    fig.update_layout(
        title=title or f"Chronological True vs Pred (channel {channel})",
        xaxis_title="Chronological row index",
        yaxis_title=f"Value (channel {channel})",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=40, r=20, t=60, b=40),
    )

    # Add a range slider (very handy for 600k timeline)
    fig.update_xaxes(
        rangeslider=dict(visible=True),
        showspikes=True,
        spikemode="across",
        spikesnap="cursor",
    )
    fig.update_yaxes(showspikes=True)

    return fig

In [54]:
fig = plot_chrono_true_pred_interactive(
    true_full, pred_full, split_mask,
    channel=4,
    max_points=200_000,
    title="TEST predictions placed back into chronological timeline (interactive)"
)
display(HTML(fig.to_html(include_plotlyjs="cdn")))

ValueError: No finite true values found in this channel after filtering.